In [1]:
!pip uninstall -y torchao
!pip install -q "diffusers>=0.27.0" transformers accelerate safetensors peft

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [4]:
"""
step1_generation_adapted.py
============================================================================
CELL 3. Generation core, adapted DIRECTLY from your working notebook
(atelectasis-gen-eta1-n20-g1.ipynb). This supersedes the earlier
reconstruction, which had three errors this file corrects:

  1. H5 stores float32 ALREADY in [-1,1]. The reconstruction divided by
     255 and rescaled, which would have fed the VAE garbage.
  2. Real config is 30 inversion / 30 inference steps, not 50/15.
  3. Inversion uses diffusers DDIMInverseScheduler, not a hand-rolled
     update. Your version is correct; mine was an approximation.

WHAT CHANGED FROM YOUR NOTEBOOK
Exactly one thing: load_pipeline() takes an optional lora_path. Every
other function is byte-equivalent in behaviour to yours -- same Welford
accumulator, same sequential N=20, same per-finding prompt, same
attention slicing, same fp16, same seed_base=0 so generators are
manual_seed(0..19).

WHY THE PROMPT STAYS LABEL-CONDITIONED
Your notebook uses prompt = f"a chest x-ray showing {finding.lower()}".
That is kept EXACTLY as-is. The fine-tuning experiment changes UNet
weights only. If the prompt convention also changed, the comparison would
confound weight adaptation with conditioning change. The prompt asymmetry
exists identically in baseline and fine-tuned arms, so it cancels.

(Note for the viva, not for this script: this means the generator does
receive class-name conditioning at inference, which is a weaker form of
label access than supervised training but not zero. Thesis 4.2 describes
the weights as label-free, which is accurate; it describes the estimator
as label-free, which is stronger than what the prompt convention
supports.)

MEASURED TIMING (from your own diagnostic cells, not estimated)
  72.4 s/image single-GPU
  154.5 s for 4 images across 2x T4  ->  ~38.6 s/image effective
  => 54 images  ~35 min
     114 images ~73 min
"""

import ast
import gc
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler, DDIMInverseScheduler
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# =============================================================================
# CONFIG -- unchanged from your notebook
# =============================================================================
H5_PATH = Path("/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_512.h5")
METADATA_CSV = Path("/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_train.csv")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_INVERSION_STEPS = 30
NUM_INFERENCE_STEPS = 30
NUM_ENSEMBLE = 20
ETA = 1
GUIDANCE_SCALE = 1.0
IMG_SIZE = 512
CHUNK_SIZE = 50

# LoRA weights directory, or None for baseline. Set before calling
# load_all_pipelines(). This is the ONLY addition to your pipeline.
LORA_PATH = None

NUM_GPUS = torch.cuda.device_count()
DEVICES = [f"cuda:{i}" for i in range(NUM_GPUS)] if NUM_GPUS > 0 else ["cpu"]


def _discover_models():
    local = []
    inp = Path("/kaggle/input")
    if inp.exists():
        for p in inp.rglob("model_index.json"):
            local.append(str(p.parent))
    return local + ["stanfordmimi/RoentGen-v2", "runwayml/stable-diffusion-v1-5"]


MODEL_CANDIDATES = _discover_models()


# =============================================================================
# Pipeline -- your loader, plus lora_path
# =============================================================================
def load_pipeline(device, lora_path=None):
    pipe = None
    for model_id in MODEL_CANDIDATES:
        try:
            is_local = Path(model_id).exists()
            kwargs = {"torch_dtype": torch.float16, "safety_checker": None,
                      "requires_safety_checker": False}
            if is_local:
                kwargs["local_files_only"] = True
            else:
                tok = globals().get("HF_TOKEN", None)
                if tok:
                    kwargs["token"] = tok
            pipe = StableDiffusionPipeline.from_pretrained(model_id, **kwargs)
            print(f"[{device}] loaded {model_id}")
            break
        except Exception as e:
            print(f"[{device}] could not load {model_id}: {e}")
    if pipe is None:
        raise RuntimeError("No candidate model could be loaded.")

    pipe = pipe.to(device)
    pipe.set_progress_bar_config(disable=True)

    # --- THE ONLY ADDITION TO YOUR PIPELINE ---
    if lora_path is not None:
        pipe.load_lora_weights(str(lora_path))
        pipe.fuse_lora()          # fuse for inference speed; no-op on baseline
        print(f"[{device}] LoRA fused from {lora_path}")
    # ------------------------------------------

    pipe.enable_attention_slicing()
    pipe.vae.enable_slicing()
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

    for m in (pipe.unet, pipe.vae, pipe.text_encoder):
        m.eval()
        for p in m.parameters():
            p.requires_grad_(False)
    return pipe


def load_all_pipelines(lora_path=None):
    pipes = {d: load_pipeline(d, lora_path) for d in DEVICES}
    scheds = {d: DDIMScheduler.from_config(pipes[d].scheduler.config) for d in DEVICES}
    inv_scheds = {d: DDIMInverseScheduler.from_config(pipes[d].scheduler.config)
                  for d in DEVICES}
    return pipes, scheds, inv_scheds


# =============================================================================
# Core -- verbatim behaviour from your notebook
# =============================================================================
@torch.no_grad()
def load_image_tensor_from_h5(h5_path, image_id):
    """H5 holds float32 (512,512) ALREADY in [-1,1]. No rescaling."""
    with h5py.File(h5_path, "r") as h5f:
        if image_id not in h5f:
            raise KeyError(f"{image_id} not found in {h5_path}")
        arr = h5f[image_id][:]
    t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
    return t.repeat(1, 3, 1, 1)


@torch.no_grad()
def encode_to_latent(pipe, image_tensor):
    image_tensor = image_tensor.to(device=pipe.device, dtype=pipe.unet.dtype)
    dist = pipe.vae.encode(image_tensor).latent_dist
    return dist.mean * pipe.vae.config.scaling_factor


@torch.no_grad()
def get_text_embeddings(pipe, prompt):
    tokens = pipe.tokenizer(
        prompt, padding="max_length", max_length=pipe.tokenizer.model_max_length,
        truncation=True, return_tensors="pt").input_ids.to(pipe.device)
    return pipe.text_encoder(tokens)[0].to(dtype=pipe.unet.dtype)


@torch.no_grad()
def ddim_invert(latents, pipe, inverse_scheduler, text_embeddings,
                num_inversion_steps=NUM_INVERSION_STEPS):
    inverse_scheduler.set_timesteps(num_inversion_steps, device=latents.device)
    latents = latents.clone()
    for t in inverse_scheduler.timesteps:
        noise_pred = pipe.unet(latents, t, encoder_hidden_states=text_embeddings).sample
        latents = inverse_scheduler.step(noise_pred, t, latents).prev_sample
    return latents


@torch.no_grad()
def ddim_sample_from_latent(z_T, pipe, scheduler, text_embeddings,
                            num_inference_steps=NUM_INFERENCE_STEPS,
                            eta=ETA, generator=None):
    scheduler.set_timesteps(num_inference_steps, device=z_T.device)
    latents = z_T.clone()
    for t in scheduler.timesteps:
        noise_pred = pipe.unet(latents, t, encoder_hidden_states=text_embeddings).sample
        latents = scheduler.step(noise_pred, t, latents, eta=eta,
                                 generator=generator).prev_sample
    return latents


@torch.no_grad()
def decode_latents_to_grayscale(pipe, latents):
    latents = latents / pipe.vae.config.scaling_factor
    image = pipe.vae.decode(latents).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    return image.float().cpu().mean(dim=1).squeeze(0)


class WelfordAccumulator:
    def __init__(self, shape, device="cpu"):
        self.n = 0
        self.mean = torch.zeros(shape, dtype=torch.float32, device=device)
        self.M2 = torch.zeros(shape, dtype=torch.float32, device=device)

    def update(self, x):
        x = x.to(dtype=torch.float32, device=self.mean.device)
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        self.M2 += delta * (x - self.mean)

    @property
    def variance(self):
        if self.n < 2:
            return torch.zeros_like(self.mean)
        return self.M2 / (self.n - 1)


@torch.no_grad()
def run_uncertainty_ensemble(pipe, scheduler, z_T, text_embeddings,
                             num_ensemble=NUM_ENSEMBLE,
                             num_inference_steps=NUM_INFERENCE_STEPS,
                             eta=ETA, seed_base=0):
    acc = WelfordAccumulator(shape=(IMG_SIZE, IMG_SIZE), device="cpu")
    for i in range(num_ensemble):
        gen = torch.Generator(device=pipe.device).manual_seed(seed_base + i)
        lat = ddim_sample_from_latent(z_T, pipe, scheduler, text_embeddings,
                                      num_inference_steps=num_inference_steps,
                                      eta=eta, generator=gen)
        img = decode_latents_to_grayscale(pipe, lat)
        acc.update(img)
        del lat, img, gen
        torch.cuda.empty_cache()
    return acc.mean, acc.variance


@torch.no_grad()
def process_one_image(pipe, scheduler, inverse_scheduler, image_id, finding, device):
    # UNCHANGED from your notebook -- prompt keeps the finding name.
    prompt = f"a chest x-ray showing {finding.lower()}"

    image_tensor = load_image_tensor_from_h5(H5_PATH, image_id)
    emb = get_text_embeddings(pipe, prompt)
    lat0 = encode_to_latent(pipe, image_tensor)
    z_T = ddim_invert(lat0, pipe, inverse_scheduler, emb,
                      num_inversion_steps=NUM_INVERSION_STEPS)
    mean_img, var_img = run_uncertainty_ensemble(
        pipe, scheduler, z_T, emb, num_ensemble=NUM_ENSEMBLE,
        num_inference_steps=NUM_INFERENCE_STEPS, eta=ETA)

    out = {"mean": mean_img.numpy().astype(np.float32),
           "variance": var_img.numpy().astype(np.float32),
           "z_T": z_T.detach().to(dtype=torch.float16).cpu().numpy(),
           "finding": finding}
    del image_tensor, emb, lat0, z_T, mean_img, var_img
    torch.cuda.empty_cache()
    gc.collect()
    return out


def process_partition(pipe, scheduler, inverse_scheduler, partition_df, device):
    out = {}
    for _, row in tqdm(partition_df.iterrows(), total=len(partition_df),
                       desc=f"[{device}]", leave=False):
        try:
            out[row["image_id"]] = process_one_image(
                pipe, scheduler, inverse_scheduler,
                row["image_id"], row["finding"], device)
        except Exception as e:
            print(f"[{device}] FAILED on {row['image_id']}: {e}")
    return out


def process_chunk(chunk_df, pipes, scheds, inv_scheds):
    parts = np.array_split(chunk_df, len(DEVICES))
    results = {}
    with ThreadPoolExecutor(max_workers=len(DEVICES)) as ex:
        futures = {ex.submit(process_partition, pipes[DEVICES[i]],
                             scheds[DEVICES[i]], inv_scheds[DEVICES[i]],
                             parts[i], DEVICES[i]): DEVICES[i]
                   for i in range(len(DEVICES)) if len(parts[i]) > 0}
        for f in as_completed(futures):
            try:
                results.update(f.result())
            except Exception as e:
                print(f"[{futures[f]}] partition failed: {e}")
    return results


def save_chunk_npz(chunk_results, chunk_idx, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    save_dict, meta = {}, []
    for image_id, d in chunk_results.items():
        save_dict[f"{image_id}__mean"] = d["mean"]
        save_dict[f"{image_id}__variance"] = d["variance"]
        save_dict[f"{image_id}__z_T"] = d["z_T"]
        meta.append({"image_id": image_id, "finding": d["finding"],
                     "variance_mean": float(d["variance"].mean()),
                     "variance_max": float(d["variance"].max())})
    p = out_dir / f"uncertainty_chunk_{chunk_idx:03d}.npz"
    np.savez_compressed(p, **save_dict)
    pd.DataFrame(meta).to_csv(
        out_dir / f"uncertainty_chunk_{chunk_idx:03d}_meta.csv", index=False)
    print(f"  saved chunk {chunk_idx}: {len(chunk_results)} imgs -> "
          f"{p.name} ({p.stat().st_size / 1e6:.1f} MB)")


def build_target_df(findings):
    md = pd.read_csv(METADATA_CSV)
    md["labels_parsed"] = md["labels"].apply(ast.literal_eval)

    def first_match(labels):
        for lab in labels:
            if lab in findings:
                return lab
        return None

    md["finding"] = md["labels_parsed"].apply(first_match)
    tdf = md[md["finding"].notna()].reset_index(drop=True)[["image_id", "finding"]]
    print(f"targeting {len(tdf):,} images across {findings}")
    print(tdf["finding"].value_counts())
    return tdf


# =============================================================================
# STEP 5a -- REPRODUCIBILITY CHECK. Run this before the full generation.
# =============================================================================
def verify_baseline_reproduces(existing_map_dir, n_check=3):
    """
    Regenerates a few images with LORA_PATH=None and compares against your
    EXISTING baseline maps.

    Why this matters: if this environment reproduces your original run,
    fine-tuned maps can be compared directly against the existing baseline
    maps and you skip regenerating 54 baseline images (~35 min saved). If
    it does NOT reproduce -- different diffusers version, different GPU
    kernel -- you must regenerate both arms with this file, or the
    comparison confounds the LoRA effect with an environment difference.
    """
    from numpy.testing import assert_allclose

    maps, _ = load_map_dir(str(existing_map_dir), method="diffusion")
    ids = list(maps.keys())[:n_check]
    print(f"Re-generating {len(ids)} image(s) at baseline to compare...\n")

    pipes, scheds, inv_scheds = load_all_pipelines(lora_path=None)
    d = DEVICES[0]
    md = pd.read_csv(METADATA_CSV)
    md["labels_parsed"] = md["labels"].apply(ast.literal_eval)

    worst = 0.0
    for image_id in ids:
        row = md[md["image_id"] == image_id]
        if row.empty:
            print(f"  {image_id}: not in metadata, skipping")
            continue
        finding = [l for l in row.iloc[0]["labels_parsed"]][0]
        res = process_one_image(pipes[d], scheds[d], inv_scheds[d],
                                image_id, finding, d)
        old, new = maps[image_id], res["variance"]
        rel = np.abs(new - old).max() / (old.max() + 1e-12)
        worst = max(worst, rel)
        corr = np.corrcoef(old.ravel(), new.ravel())[0, 1]
        print(f"  {image_id}: max rel diff {rel:.4f}, corr {corr:.6f}")

    print(f"\n{'=' * 62}")
    if worst < 0.01:
        print("REPRODUCES. Skip baseline regeneration -- compare fine-tuned")
        print("maps directly against your existing Stage 9 baseline maps.")
    else:
        print(f"DOES NOT REPRODUCE (worst rel diff {worst:.4f}).")
        print("You MUST regenerate both arms with this file. Budget 2x the")
        print("generation time. Do not compare against the original maps.")
    print("=" * 62)
    return worst


if __name__ == "__main__" and not globals().get("DRIVER_MODE", False):
    print(f"devices: {DEVICES}")
    print(f"models:  {MODEL_CANDIDATES[:2]}")
    print("\nMeasured timing from your diagnostics: ~38.6 s/image on 2x T4")
    print("  54 images  -> ~35 min")
    print("  114 images -> ~73 min")
    print("\nNext: run verify_baseline_reproduces(<your existing map dir>)")
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [5]:
DRIVER_MODE = True
"""
step2_caption_manifest.py
============================================================================
CELL 4. Builds the LoRA training manifest. ~2 minutes, no GPU.

CORRECTED against your actual notebook: METADATA_CSV is
vindr_cxr_train.csv (the file your generator reads), whose `labels` column
is a Python-list string parsed with ast.literal_eval.

THE TRAINING CAPTION IS LABEL-FREE. THE GENERATION PROMPT IS NOT.
That asymmetry is deliberate and it is the crux of the experiment's
validity.

Your generator conditions on the finding:
    prompt = f"a chest x-ray showing {finding.lower()}"

Training captions here do NOT:
    caption = "a chest x-ray"

Why:
  - Fine-tuning adapts the APPEARANCE PRIOR -- acquisition
    characteristics, contrast, VinDr's preprocessing. That is the domain
    gap being tested.
  - If training captions carried class names, the LoRA weights would
    encode pathology-label information, and any localisation change after
    fine-tuning would confound domain adaptation with newly injected
    label supervision. Uninterpretable.
  - Generation prompts stay exactly as your notebook has them, so the
    prompt convention is IDENTICAL across baseline and fine-tuned arms
    and cancels in the paired comparison.

Net effect: the only difference between arms is UNet weights adapted on
label-free in-domain images. Clean test.

TRAIN SPLIT ONLY. The test images the thesis evaluates on must stay unseen
by the fine-tuned generator.
"""

from pathlib import Path

import pandas as pd

METADATA_CSV = Path("/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_train.csv")

# File carrying the `split` column (the one stage9_final.py reads as
# METADATA_CSV_PATH). If vindr_cxr_train.csv is already train-only, set
# SPLIT_SOURCE_CSV = None and the filter is skipped.
SPLIT_SOURCE_CSV = Path("/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_metadata.csv")

OUTPUT_CSV = Path("/kaggle/working/train_manifest.csv")

N_TRAIN_IMAGES = 2000
CAPTION = "a chest x-ray"        # label-free, identical for every row
RANDOM_SEED = 42


def main():
    df = pd.read_csv(METADATA_CSV)
    print(f"{METADATA_CSV.name}: {len(df)} rows")

    test_ids = set()
    if SPLIT_SOURCE_CSV is not None and Path(SPLIT_SOURCE_CSV).exists():
        sdf = pd.read_csv(SPLIT_SOURCE_CSV)
        if "split" in sdf.columns:
            test_ids = set(sdf[sdf["split"] == "test"]["image_id"].astype(str))
            train_ids = set(sdf[sdf["split"] == "train"]["image_id"].astype(str))
            before = len(df)
            df = df[df["image_id"].astype(str).isin(train_ids)]
            print(f"  train-split filter: {before} -> {len(df)} rows "
                  f"({len(test_ids)} test ids known)")
        else:
            print("  WARNING: no 'split' column found; assuming train-only input")
    else:
        print("  no split source configured; assuming train-only input")

    n = min(N_TRAIN_IMAGES, len(df))
    if n < N_TRAIN_IMAGES:
        print(f"  NOTE: only {n} images available, using all")
    sample = df.sample(n=n, random_state=RANDOM_SEED).copy()
    sample["caption"] = CAPTION

    out = sample[["image_id", "caption"]]
    out.to_csv(OUTPUT_CSV, index=False)

    assert out["caption"].nunique() == 1, "captions must be identical"
    leak = set(out["image_id"].astype(str)) & test_ids
    assert not leak, f"LEAK: {len(leak)} training images are in the test split"

    print(f"\nwrote {len(out)} rows -> {OUTPUT_CSV}")
    print(f"caption (identical for all): {CAPTION!r}")
    print(f"unique captions:    {out['caption'].nunique()}   [must be 1]")
    print(f"test-split leakage: {len(leak)}   [must be 0]")

    if not test_ids:
        print("\n  WARNING: no test ids loaded, so the leakage assertion was")
        print("  vacuous. Point SPLIT_SOURCE_CSV at the file with the")
        print("  `split` column and re-run before training.")
    return out


if __name__ == "__main__" and not globals().get("DRIVER_MODE", False):
    main()
main()

vindr_cxr_train.csv: 3075 rows
  train-split filter: 3075 -> 3075 rows (660 test ids known)

wrote 2000 rows -> /kaggle/working/train_manifest.csv
caption (identical for all): 'a chest x-ray'
unique captions:    1   [must be 1]
test-split leakage: 0   [must be 0]


,image_id,caption
718,3cdcd43f230d185be9a3ea2463f348dc,a chest x-ray
2442,ca2a64d3c50654de6327f3880bbc17d3,a chest x-ray
2484,ce6877e88fc20990497099e6196b6ae4,a chest x-ray
1326,7225983eb43be7f2dcb9e75381a8f3e5,a chest x-ray
1640,8af84125f707b873c506f65de72e02e4,a chest x-ray
...,...,...
1287,6e4391555899c8474c4d32f42b2ba21b,a chest x-ray
1920,a137951bfa9be68fec6cb6ef0a679d20,a chest x-ray
1667,8d71580638dabfe93678f1a7bab30bfe,a chest x-ray
2652,dbca1fb4fbbfbd6032a9764f3e922ad3,a chest x-ray


In [6]:
DRIVER_MODE = True
"""
step3_lora_training.py
============================================================================
CELL 5. LoRA fine-tuning of the diffusion UNet. ~2h15 target.

CORRECTED against your actual notebook. The critical fix:

    YOUR H5 STORES float32 ALREADY NORMALIZED TO [-1, 1].

The earlier draft divided by 255 and then rescaled to [-1,1], which would
have fed the VAE values around -1.0 for every pixel and trained the LoRA
on noise. This version replicates load_image_tensor_from_h5() from your
generator exactly: read the array, replicate to 3 channels, done.

Model loading also matches your notebook: scan /kaggle/input recursively
for model_index.json, prefer local weights, fall back to the Hub.

CHECKPOINTS EVERY 250 STEPS. If the session dies or training stalls you
keep the best checkpoint so far, and a partially adapted LoRA still gives
a valid before/after provided reconstruction fidelity moved.

GO/NO-GO GATE AT STEP 250: printed automatically. Loss flat or rising ->
kill, set LEARNING_RATE = 5e-5, restart. Budget allows one restart.
"""

import time
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# =============================================================================
# CONFIG
# =============================================================================
H5_PATH = Path("/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_512.h5")
MANIFEST_CSV = Path("/kaggle/working/train_manifest.csv")
OUTPUT_DIR = Path("/kaggle/working/lora_cxr")

LORA_RANK = 8
LORA_ALPHA = 8
LEARNING_RATE = 1e-4
LR_WARMUP_STEPS = 100
BATCH_SIZE = 2
GRAD_ACCUM = 4               # effective batch 8
MAX_STEPS = 1500
CHECKPOINT_EVERY = 250
SEED = 42
DEVICE = "cuda:0"            # train on one GPU; generation uses both later


def _discover_models():
    local = []
    inp = Path("/kaggle/input")
    if inp.exists():
        for p in inp.rglob("model_index.json"):
            local.append(str(p.parent))
    return local + ["stanfordmimi/RoentGen-v2", "runwayml/stable-diffusion-v1-5"]


MODEL_CANDIDATES = _discover_models()


# =============================================================================
# Dataset -- mirrors load_image_tensor_from_h5() from your generator
# =============================================================================
class H5CXRDataset(Dataset):
    def __init__(self, manifest_csv, h5_path, tokenizer):
        self.df = pd.read_csv(manifest_csv)
        self.h5_path = str(h5_path)
        self._h5 = None

        assert self.df["caption"].nunique() == 1, \
            "captions must be identical -- see step2 docstring"
        caption = self.df["caption"].iloc[0]
        tok = tokenizer(caption, padding="max_length",
                        max_length=tokenizer.model_max_length,
                        truncation=True, return_tensors="pt")
        self.input_ids = tok.input_ids[0]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        image_id = str(self.df.iloc[idx]["image_id"])
        # ALREADY [-1,1] float32. No division, no rescaling.
        arr = self._h5[image_id][:]
        x = torch.from_numpy(arr).unsqueeze(0).repeat(3, 1, 1)
        return {"pixel_values": x, "input_ids": self.input_ids}


def _load_components():
    from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
    from transformers import CLIPTextModel, CLIPTokenizer

    last_err = None
    for model_id in MODEL_CANDIDATES:
        try:
            kw = {"local_files_only": True} if Path(model_id).exists() else {}
            tokenizer = CLIPTokenizer.from_pretrained(
                model_id, subfolder="tokenizer", **kw)
            text_encoder = CLIPTextModel.from_pretrained(
                model_id, subfolder="text_encoder", **kw).to(DEVICE, torch.float16)
            vae = AutoencoderKL.from_pretrained(
                model_id, subfolder="vae", **kw).to(DEVICE, torch.float16)
            unet = UNet2DConditionModel.from_pretrained(
                model_id, subfolder="unet", **kw).to(DEVICE, torch.float32)
            sched = DDPMScheduler.from_pretrained(
                model_id, subfolder="scheduler", **kw)
            print(f"loaded components from {model_id}")
            return tokenizer, text_encoder, vae, unet, sched
        except Exception as e:
            last_err = e
            print(f"  could not load {model_id}: {e}")
    raise RuntimeError(f"no model could be loaded; last error: {last_err}")


def _lora_state_dict(unet):
    from peft.utils import get_peft_model_state_dict
    return get_peft_model_state_dict(unet)


def _save(unet, path):
    from diffusers import StableDiffusionPipeline
    Path(path).mkdir(parents=True, exist_ok=True)
    StableDiffusionPipeline.save_lora_weights(
        save_directory=str(path), unet_lora_layers=_lora_state_dict(unet))


def main():
    from peft import LoraConfig

    torch.manual_seed(SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    tokenizer, text_encoder, vae, unet, noise_scheduler = _load_components()

    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)
    unet.requires_grad_(False)

    unet.add_adapter(LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, init_lora_weights="gaussian",
        target_modules=["to_q", "to_k", "to_v", "to_out.0"]))

    params = [p for p in unet.parameters() if p.requires_grad]
    n_train = sum(p.numel() for p in params)
    n_total = sum(p.numel() for p in unet.parameters())
    print(f"trainable: {n_train:,} / {n_total:,} "
          f"({100 * n_train / n_total:.3f}% of UNet)")

    ds = H5CXRDataset(MANIFEST_CSV, H5_PATH, tokenizer)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=2, drop_last=True)
    print(f"dataset: {len(ds)} images\n")

    opt = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=1e-2)
    scaler = torch.cuda.amp.GradScaler()

    unet.train()
    step, t0, running = 0, time.time(), []
    pbar = tqdm(total=MAX_STEPS)

    while step < MAX_STEPS:
        for batch in dl:
            if step >= MAX_STEPS:
                break

            with torch.no_grad():
                px = batch["pixel_values"].to(DEVICE, torch.float16)
                lat = vae.encode(px).latent_dist.sample()
                lat = (lat * vae.config.scaling_factor).float()
                emb = text_encoder(batch["input_ids"].to(DEVICE))[0].float()

            noise = torch.randn_like(lat)
            ts = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                               (lat.shape[0],), device=DEVICE).long()
            noisy = noise_scheduler.add_noise(lat, noise, ts)

            with torch.cuda.amp.autocast():
                pred = unet(noisy, ts, encoder_hidden_states=emb).sample
                loss = F.mse_loss(pred.float(), noise.float()) / GRAD_ACCUM

            scaler.scale(loss).backward()
            running.append(loss.item() * GRAD_ACCUM)

            if (step + 1) % GRAD_ACCUM == 0:
                warm = min(1.0, (step + 1) / max(1, LR_WARMUP_STEPS))
                for g in opt.param_groups:
                    g["lr"] = LEARNING_RATE * warm
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()

            step += 1
            pbar.update(1)
            if step % 50 == 0:
                pbar.set_postfix({"loss": f"{np.mean(running[-50:]):.4f}",
                                  "min": f"{(time.time() - t0) / 60:.0f}"})

            if step % CHECKPOINT_EVERY == 0:
                _save(unet, OUTPUT_DIR / f"checkpoint-{step}")
                print(f"\n  checkpoint-{step} saved "
                      f"(loss {np.mean(running[-CHECKPOINT_EVERY:]):.4f})")

                if step == 250:
                    first, latest = float(np.mean(running[:100])), float(np.mean(running[-100:]))
                    print("\n" + "=" * 62)
                    print(f"GATE @ 250: loss {first:.4f} -> {latest:.4f} "
                          f"({100 * (latest - first) / first:+.1f}%)")
                    if latest >= first:
                        print("  NOT CONVERGING. Kill this run, set")
                        print("  LEARNING_RATE = 5e-5, restart. One restart only.")
                    else:
                        print("  Converging. Continue.")
                    print("=" * 62 + "\n")

    pbar.close()
    _save(unet, OUTPUT_DIR)
    print(f"\ndone in {(time.time() - t0) / 60:.1f} min -> {OUTPUT_DIR}")
    print("\nSTEP 4 NOW: save OUTPUT_DIR as a Kaggle Dataset before running")
    print("anything else. /kaggle/working does not persist.")


if __name__ == "__main__" and not globals().get("DRIVER_MODE", False):
    main()
main()

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

loaded components from /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete
trainable: 1,659,904 / 867,570,628 (0.191% of UNet)
dataset: 2000 images



/tmp/ipykernel_59/433402279.py:165: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


  0%|          | 0/1500 [00:00<?, ?it/s]

/tmp/ipykernel_59/433402279.py:187: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



  checkpoint-250 saved (loss 0.1058)

GATE @ 250: loss 0.1078 -> 0.0985 (-8.6%)
  Converging. Continue.


  checkpoint-500 saved (loss 0.0918)

  checkpoint-750 saved (loss 0.1014)

  checkpoint-1000 saved (loss 0.0922)

  checkpoint-1250 saved (loss 0.1138)

  checkpoint-1500 saved (loss 0.1145)

done in 15.5 min -> /kaggle/working/lora_cxr

STEP 4 NOW: save OUTPUT_DIR as a Kaggle Dataset before running
anything else. /kaggle/working does not persist.


In [7]:
!kaggle datasets init -p /kaggle/working/

Data package template written to: /kaggle/working/dataset-metadata.json


In [15]:
import json, os, shutil, subprocess
from pathlib import Path

# ---------------- config ----------------
WORKING       = Path("/kaggle/working")
DATASET_SLUG  = "roentgen-lora-cxr-rank8"      # lowercase, hyphens, 3-50 chars
DATASET_TITLE = "RoentGen-v2 LoRA CXR rank8"   # 6-50 chars
STAGING       = Path("/kaggle/_ds_staging")    # OUTSIDE /kaggle/working on purpose
# ----------------------------------------

# 1. credentials from Secrets
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
username = sec.get_secret("KAGGLE_USERNAME")
key      = sec.get_secret("KAGGLE_KEY")

kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
(kdir / "kaggle.json").write_text(json.dumps({"username": username, "key": key}))
(kdir / "kaggle.json").chmod(0o600)
os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = username, key
print(f"credentials set for {username}")

# 2. copy EVERYTHING from /kaggle/working
if STAGING.exists():
    shutil.rmtree(STAGING)
shutil.copytree(WORKING, STAGING)

# drop junk that shouldn't ship
for pattern in ["**/__pycache__", "**/.ipynb_checkpoints", "**/.git"]:
    for p in STAGING.glob(pattern):
        shutil.rmtree(p, ignore_errors=True)

total = 0
print(f"\nstaged from {WORKING}:")
for p in sorted(STAGING.rglob("*")):
    if p.is_file():
        mb = p.stat().st_size / 1e6
        total += mb
        print(f"  {p.relative_to(STAGING)}  {mb:.2f} MB")
print(f"\ntotal: {total:.1f} MB  ({sum(1 for p in STAGING.rglob('*') if p.is_file())} files)")
assert total > 0, "/kaggle/working is empty"

# 3. metadata
meta = {"title": DATASET_TITLE,
        "id": f"{username}/{DATASET_SLUG}",
        "licenses": [{"name": "CC0-1.0"}]}
(STAGING / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
print(f"id: {meta['id']}")

# 4. create, or version if it already exists
def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout or "", r.stderr or "")
    return r.returncode

if run(f'kaggle datasets create -p "{STAGING}" --dir-mode zip') != 0:
    print("create failed — trying as a new version")
    run(f'kaggle datasets version -p "{STAGING}" -m "step3 lora + manifest" --dir-mode zip')

print(f"\nhttps://www.kaggle.com/datasets/{username}/{DATASET_SLUG}")

credentials set for kartikichandratre

staged from /kaggle/working:
  .virtual_documents/__notebook_source__.ipynb  0.03 MB
  _ds_staging/checkpoint-1000/pytorch_lora_weights.safetensors  6.68 MB
  _ds_staging/checkpoint-1250/pytorch_lora_weights.safetensors  6.68 MB
  _ds_staging/checkpoint-1500/pytorch_lora_weights.safetensors  6.68 MB
  _ds_staging/checkpoint-250/pytorch_lora_weights.safetensors  6.68 MB
  _ds_staging/checkpoint-500/pytorch_lora_weights.safetensors  6.68 MB
  _ds_staging/checkpoint-750/pytorch_lora_weights.safetensors  6.68 MB
  _ds_staging/dataset-metadata.json  0.00 MB
  _ds_staging/pytorch_lora_weights.safetensors  6.68 MB
  dataset-metadata.json  0.00 MB
  lora_cxr/checkpoint-1000/pytorch_lora_weights.safetensors  6.68 MB
  lora_cxr/checkpoint-1250/pytorch_lora_weights.safetensors  6.68 MB
  lora_cxr/checkpoint-1500/pytorch_lora_weights.safetensors  6.68 MB
  lora_cxr/checkpoint-250/pytorch_lora_weights.safetensors  6.68 MB
  lora_cxr/checkpoint-500/pytorch_lora

In [16]:
LORA_PATH = "/kaggle/working/lora_cxr"
pipes, scheds, inv_scheds = load_all_pipelines(lora_path=LORA_PATH)
tdf = build_target_df(["Consolidation"])
d = DEVICES[0]
res = process_one_image(pipes[d], scheds[d], inv_scheds[d],
                        tdf.iloc[0]["image_id"], tdf.iloc[0]["finding"], d)
print("variance range:", res["variance"].min(), res["variance"].max())
print("all-zero?", np.allclose(res["variance"], 0))

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

[cuda:0] loaded /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


[cuda:0] LoRA fused from /kaggle/working/lora_cxr


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

[cuda:1] loaded /kaggle/input/datasets/pgc17ms072/roentgen-v2-complete


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


[cuda:1] LoRA fused from /kaggle/working/lora_cxr
targeting 254 images across ['Consolidation']
finding
Consolidation    254
Name: count, dtype: int64
variance range: 4.980578e-06 0.10306793
all-zero? False
